In [ ]:
# %% Deep learning - Section 22.205
#    Transferring the screaming bathtub

# This code pertains a deep learning course provided by Mike X. Cohen on Udemy:
#   > https://www.udemy.com/course/deeplearning_x
# The "base" code in this repository is adapted (with very minor modifications)
# from code developed by the course instructor (Mike X. Cohen), while the
# "exercises" and the "code challenges" contain more original solutions and
# creative input from my side. If you are interested in DL (and if you are
# reading this statement, chances are that you are), go check out the course, it
# is singularly good.

In [1]:
# %% Libraries and modules
import numpy                  as np
import matplotlib.pyplot      as plt
import torch
import torch.nn               as nn
import seaborn                as sns
import copy
import torch.nn.functional    as F
import pandas                 as pd
import scipy.stats            as stats
import sklearn.metrics        as skm
import time
import sys
import imageio.v2
import torchvision
import torchvision.transforms as T

from torch.utils.data                 import DataLoader,TensorDataset,Dataset,Subset
from sklearn.model_selection          import train_test_split
from google.colab                     import files
from torchsummary                     import summary
from scipy.stats                      import zscore
from sklearn.decomposition            import PCA
from scipy.signal                     import convolve2d
from torchsummary                     import summary
from matplotlib.gridspec              import GridSpec
from IPython                          import display
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')
plt.style.use('default')


In [ ]:
# %% Import and freeze VGG-19

# Import the model
vgg_net = torchvision.models.vgg19(weights=True)

# Freeze all the layers
for p in vgg_net.parameters():
    p.requires_grad = False

# Switch to evaluation mode
vgg_net.eval()


In [ ]:
# %% Use GPU and ship model there

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

vgg_net.to(device)


In [ ]:
# %% Import images

uploaded = files.upload()

for fn in uploaded.keys():
    print('User uploaded file "{name}" with length {length} bytes'.format(
        name=fn, length=len(uploaded[fn])))


In [ ]:
# %% Load images

img_content = imageio.v2.imread('content_image_3.jpeg')
img_style   = imageio.v2.imread('style_image_3b.jpeg')

# Initialize the target image as random numbers
img_target = np.random.randint(low=0,high=255,size=img_content.shape,dtype=np.uint8)
# img_target = img_content

# Check sizes
print("Content img:")
print(img_content.shape)
print("\nTarget img:")
print(img_target.shape)
print("\nStyle img:")
print(img_style.shape)


In [ ]:
# %% Reduce image sizes to match the model (also, would take too long)

# Transforms (resize and normalise)
m  = [0.485, 0.456, 0.406]
s  = [0.229, 0.224, 0.225]

Ts = T.Compose([ T.ToTensor(),
                 T.Resize(256),
                 T.Normalize(mean=m,std=s) ])

# Apply transforms (unsqueeze to turn into 4D tensor and ship to GPU)
img_content = Ts( img_content ).unsqueeze(0).to(device)
img_style   = Ts( img_style   ).unsqueeze(0).to(device)
img_target  = Ts( img_target  ).unsqueeze(0).to(device)

# Check sizes
print("Content img:")
print(img_content.shape)
print("\nTarget img:")
print(img_target.shape)
print("\nStyle img:")
print(img_style.shape)


In [ ]:
# %% Plotting

# Plot the "before" pics
phi = (1 + np.sqrt(5)) / 2
fig,ax = plt.subplots(1,3,figsize=(phi*10,10))

pic = img_content.cpu().squeeze().numpy().transpose((1,2,0))
pic = (pic-np.min(pic)) / (np.max(pic)-np.min(pic))
ax[0].imshow(pic)
ax[0].set_title('Content picture')

pic = img_target.cpu().squeeze().numpy().transpose((1,2,0))
pic = (pic-np.min(pic)) / (np.max(pic)-np.min(pic))
ax[1].imshow(pic)
ax[1].set_title('Target picture')

pic = img_style.cpu().squeeze().numpy().transpose((1,2,0))
pic = (pic-np.min(pic)) / (np.max(pic)-np.min(pic))
ax[2].imshow(pic)
ax[2].set_title('Style picture')

plt.savefig('figure1_transferring_screaming.png')
plt.show()
files.download('figure1_transferring_screaming.png')


In [8]:
# Function to returns feature maps

def get_feature_map_activations(img,net):

    # Prealloacte feature maps lists
    feature_maps  = []
    feature_names = []

    conv_layer_idx = 0

    # Loop through all layers in the features block (the conv block)
    for layer_num in range(len(net.features)):

        # Print out info from this layer
        # print(layer_num,net.features[layer_num])

        # Process the image through this layer
        img = net.features[layer_num](img)

        # Store the image if it is a conv2d layer
        if 'Conv2d' in str(net.features[layer_num]):
            feature_maps.append( img )
            feature_names.append( 'Conv_layer_' + str(conv_layer_idx) )
            conv_layer_idx += 1

    return feature_maps,feature_names


In [9]:
# Function to compute the Gram matrix of the feature activation map

def gram_matrix(M):

    # Reshape to 2D
    _,chans,height,width = M.shape
    M = M.reshape(chans,height*width)

    # Compute covariance matrix (scale by 1/n)
    gram = torch.mm(M,M.t()) / (chans*height*width)

    return gram


In [ ]:
# Inspect the output of the feature maps function

feat_maps,feat_names = get_feature_map_activations(img_content,vgg_net)

# Print some info
for i in range(len(feat_names)):
    print('Feature map "%s" is size %s'%(feat_names[i],(feat_maps[i].shape)))


In [ ]:
# %% Plotting

# Inspect the content image
content_feature_maps,content_feature_names = get_feature_map_activations(img_content,vgg_net)

phi = (1 + np.sqrt(5)) / 2
fig,axs = plt.subplots(2,5,figsize=(phi*10,10))
for i in range(5):

    # Average over all feature maps from this layer and normalise
    pic = np.mean( content_feature_maps[i].cpu().squeeze().numpy(),axis=0 )
    pic = (pic-np.min(pic)) / (np.max(pic)-np.min(pic))

    axs[0,i].imshow(pic,cmap='gray')
    axs[0,i].set_title('Content layer ' + str(content_feature_names[i]))

    # Plot the gram matrix
    pic = gram_matrix(content_feature_maps[i]).cpu().numpy()
    pic = (pic-np.min(pic)) / (np.max(pic)-np.min(pic))

    axs[1,i].imshow(pic,cmap='gray',vmax=.1)
    axs[1,i].set_title('Gram matrix, layer ' + str(content_feature_names[i]))

plt.tight_layout()

plt.savefig('figure2_transferring_screaming.png')
plt.show()
files.download('figure2_transferring_screaming.png')


In [ ]:
# Plotting

# Inspect the style image
style_feature_maps,style_feature_names = get_feature_map_activations(img_style,vgg_net)

phi = (1 + np.sqrt(5)) / 2
fig,axs = plt.subplots(2,5,figsize=(phi*10,10))

for i in range(5):

    # Average over all feature maps from this layer and normalise
    pic = np.mean( style_feature_maps[i].cpu().squeeze().numpy(),axis=0 )
    pic = (pic-np.min(pic)) / (np.max(pic)-np.min(pic))

    axs[0,i].imshow(pic,cmap='gray')
    axs[0,i].set_title('Content layer ' + str(style_feature_names[i]))

    # Plot the gram matrix
    pic = gram_matrix(style_feature_maps[i]).cpu().numpy()
    pic = (pic-np.min(pic)) / (np.max(pic)-np.min(pic))

    axs[1,i].imshow(pic,cmap='gray',vmax=.1)
    axs[1,i].set_title('Gram matrix, layer ' + str(style_feature_names[i]))

plt.tight_layout()

plt.savefig('figure3_transferring_screaming.png')
plt.show()
files.download('figure3_transferring_screaming.png')


In [245]:
# %% Pick meta-parameters for the transfer

# Layers to use (bathtub)
layers4content = [ 'Conv_layer_1','Conv_layer_4' ]
layers4style   = [ 'Conv_layer_1','Conv_layer_2','Conv_layer_3','Conv_layer_4','Conv_layer_5' ]
weights4style  = [      1       ,     .5      ,     .5      ,     .2      ,     .1       ]

# Layers to use (beauvais)
layers4content = [ 'Conv_layer_1','Conv_layer_4' ]
layers4style   = [ 'Conv_layer_1','Conv_layer_2','Conv_layer_3','Conv_layer_4','Conv_layer_5' ]
weights4style  = [     .7       ,     .5      ,     .5      ,     .2      ,     .4       ]

# Layers to use (dog - stars)
layers4content = [ 'Conv_layer_2','Conv_layer_4' ]
layers4style   = [ 'Conv_layer_1','Conv_layer_2','Conv_layer_3','Conv_layer_4','Conv_layer_5' ]
weights4style  = [      .8       ,     .8      ,     .6      ,     .4      ,     .2       ]

# style_scaling = 3e5
# optimizer = torch.optim.RMSprop([target],lr=.03)

# Layers to use (dog - picasso)
layers4content = [ 'Conv_layer_2','Conv_layer_4' ]
layers4style   = [ 'Conv_layer_1','Conv_layer_2','Conv_layer_3','Conv_layer_4','Conv_layer_5','Conv_layer_6','Conv_layer_7','Conv_layer_8' ]
weights4style  = [     .2       ,     .3      ,     .4      ,      .6      ,       .8,             .6      ,      .2      ,     .1       ]

layers4content = ['Conv_layer_5']
layers4style   = [ 'Conv_layer_1','Conv_layer_2','Conv_layer_3','Conv_layer_5','Conv_layer_7' ]
weights4style  = [      1.0,            0.9,           0.8,           0.5,          0.2    ]

# style_scaling = 3e5
# optimizer = torch.optim.RMSprop([target],lr=.03)

# Layers to use (portrait)
layers4content = [ 'Conv_layer_1','Conv_layer_2' ]
layers4style   = [ 'Conv_layer_0','Conv_layer_1','Conv_layer_2' ]
weights4style  = [     .8       ,       1       ,     .4        ]

# Layers to use (paris - stars)
layers4content = [ 'Conv_layer_1','Conv_layer_4' ]
layers4style   = [ 'Conv_layer_1','Conv_layer_2','Conv_layer_3','Conv_layer_4','Conv_layer_5' ]
weights4style  = [      1       ,     .8      ,     .6      ,     .4      ,     .2       ]

# Layers to use (paris - auvers)
layers4content = [ 'Conv_layer_1','Conv_layer_4' ]
layers4style   = [ 'Conv_layer_5','Conv_layer_6','Conv_layer_7','Conv_layer_8','Conv_layer_9' ]
weights4style  = [     .7       ,     .8      ,     .9      ,     .2      ,     .1       ]


In [246]:
# %% Run the style transfer

# Copy the target image and ship to GPU
target = img_target.clone()
target.requires_grad = True

target = target.to(device)
style_scaling = 1e6
style_scaling = 3e5

# Content feature maps
feat_maps,feat_names = get_feature_map_activations(img_content,vgg_net)

# Number training epochs
numepochs = 1500
numepochs = 5000

# Optimizer for backpropagation (RMSprop is the recommended one)
optimizer = torch.optim.RMSprop([target],lr=.03)

losses_style   = []
losses_content = []
frames         = []

for epoch_i in range(numepochs):

    # Save initial frame
    if epoch_i == 0:
        frames.append(target.detach().cpu().clone())

    # Get the target feature maps
    target_feature_maps,target_feature_names = get_feature_map_activations(target,vgg_net)

    # Preallocate individual loss components as tensors to ensure combined_loss is also a tensor
    style_loss   = torch.tensor(0.0,device=device)
    content_loss = torch.tensor(0.0,device=device)

    # Loop over layers
    for layer_i in range(len(target_feature_names)):

        # compute the content loss (MSE)
        if target_feature_names[layer_i] in layers4content:
            content_loss += torch.mean((target_feature_maps[layer_i] - content_feature_maps[layer_i])**2)

        # Compute the style loss (MSE)
        if target_feature_names[layer_i] in layers4style:

            # Gram matrices
            G_target = gram_matrix(target_feature_maps[layer_i])
            G_style  = gram_matrix(style_feature_maps[layer_i])

            # Compute loss (de-weighted with increasing depth)
            style_loss += torch.mean((G_target - G_style)**2) * weights4style[layers4style.index(target_feature_names[layer_i])]

    # Store separate losses
    losses_style.append(style_loss.item())
    losses_content.append(content_loss.item())

    # Combine losses (scale style losses)
    combined_loss = style_scaling*style_loss + content_loss

    # Backpropagation
    optimizer.zero_grad()
    combined_loss.backward()
    optimizer.step()

    # Store every 100 epochs
    if epoch_i % 100 == 0:
        frames.append(target.detach().cpu().clone())

# Save final frame
frames.append(target.detach().cpu().clone())


In [ ]:
# %% Plotting

# Plot the "after" pics
phi = (1 + np.sqrt(5)) / 2
fig = plt.figure(figsize=(phi*10,10))

gs = GridSpec(1,3,width_ratios=[1,1,0.6])
ax = [fig.add_subplot(gs[i]) for i in range(3)]

pic = img_content.cpu().squeeze().numpy().transpose((1,2,0))
pic = (pic-np.min(pic)) / (np.max(pic)-np.min(pic))
ax[0].imshow(pic)
ax[0].set_title('Content picture',fontweight='bold')
ax[0].set_xticks([])
ax[0].set_yticks([])

pic = torch.sigmoid(target).cpu().detach().squeeze().numpy().transpose((1,2,0))
ax[1].imshow(pic)
ax[1].set_title('Target picture',fontweight='bold')
ax[1].set_xticks([])
ax[1].set_yticks([])

pic = img_style.cpu().squeeze().numpy().transpose((1,2,0))
pic = (pic-np.min(pic)) / (np.max(pic)-np.min(pic))
ax[2].imshow(pic)
ax[2].set_title('Style picture',fontweight='bold')
ax[2].set_xticks([])
ax[2].set_yticks([])

plt.savefig('figure4_transferring_screaming.png')
plt.show()
files.download('figure4_transferring_screaming.png')


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5)) / 2
fig = plt.figure(figsize=(phi*10,10))

upsampled = F.interpolate(torch.sigmoid(target),scale_factor=12,mode='bilinear',align_corners=False)
pic       = upsampled.cpu().detach().squeeze().numpy().transpose((1,2,0))

plt.imshow(pic)
plt.axis('off')

plt.subplots_adjust(left=0, right=1, top=1, bottom=0)

plt.savefig('figure5_transferring_screaming.png',bbox_inches='tight',pad_inches=0)
plt.show()
files.download('figure5_transferring_screaming.png')


In [ ]:
# %% Exercise 1
#    The minimization loss has two components (style and content). Modify the code to store these two components in a
#    Nx2 matrix (for N training epochs). Then plot them. This will help you understand and adjust the styleScaling gain
#    factor.

# The style loss is much smaller, of about 6 orders of magnitude, hence the
# parameter style_scaling = 1e6

# Plotting
phi = (1 + np.sqrt(5)) / 2
fig, ax1 = plt.subplots(figsize=(phi*5,5))

ax1.plot(losses_content, '-', label='content',color='tab:blue')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Content Loss (MSE)')

ax2 = ax1.twinx()
ax2.plot(losses_style, '-', label='raw style',color='tab:red')
ax2.set_ylabel('Raw style Loss (MSE)')

lines_1,labels_1 = ax1.get_legend_handles_labels()
lines_2,labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1+lines_2, labels_1+labels_2)

plt.title('Style and Content Loss')

plt.savefig('figure40_transferring_screaming_extra1.png')
plt.show()
files.download('figure40_transferring_screaming_extra1.png')


In [250]:
# %% Exercise 2
#    Change the layers for minimizing losses to content and style images. Do you notice an effect of minimizing the
#    earlier vs. later layers? How about more vs. fewer layers?

# Quite cool! Using later layers reduces the details from the content and puts
# more style on the target; fewer early layers do basically the opposite; more
# layers in general kind of produce an intermediate nice result between the two
# previous attempts

# Layers to use (bathtub) - later
layers4content = [ 'Conv_layer_1','Conv_layer_4' ]
layers4style   = [ 'Conv_layer_5','Conv_layer_6','Conv_layer_7','Conv_layer_8','Conv_layer_9' ]
weights4style  = [      1       ,     .5      ,     .5      ,     .2      ,     .1       ]

# Layers to use (bathtub) - fewer
layers4content = [ 'Conv_layer_1','Conv_layer_4' ]
layers4style   = [ 'Conv_layer_1','Conv_layer_2' ]
weights4style  = [      1       ,     .5       ]

# Layers to use (bathtub) - more
layers4content = [ 'Conv_layer_1','Conv_layer_4' ]
layers4style   = [ 'Conv_layer_1','Conv_layer_2','Conv_layer_3','Conv_layer_4','Conv_layer_5','Conv_layer_6','Conv_layer_7','Conv_layer_8' ]
weights4style  = [      1       ,     .5      ,     .5      ,      .2      ,       .3,             .3      ,      .2      ,     .1       ]


In [248]:
# %% Exercise 3
#    It's pretty neat to see the target image evolve over time. Modify the code to save the target image every, e.g.,
#    100 epochs. Then you can make a series of images showing how the noise transforms into a lovely picture.

# Just mesmerising.

# Produce movie
def tensor_to_image(t):
    img = torch.sigmoid(t).squeeze().permute(1,2,0).numpy()
    return img

frames_np = [tensor_to_image(f) for f in frames]


In [ ]:
# %% Exercise 3
#    Continue ...

import matplotlib.animation as animation

phi = (1 + np.sqrt(5)) / 2
fig, ax = plt.subplots(figsize=(phi*6,6))
ax.axis('off')

plt.subplots_adjust(left=0, right=1, top=1, bottom=0)

im = ax.imshow(frames_np[0])

def update(i):
    im.set_array(frames_np[i])
    return (im,)

# Interval is ms between frames
animat = animation.FuncAnimation(
    fig,
    update,
    frames=len(frames_np),
    interval=200 )

animat.save('figure47_transferring_screaming_extra3_video.gif',writer='pillow',fps=5)
files.download('figure47_transferring_screaming_extra3_video.gif')

animat.save('figure47_transferring_screaming_extra3_video.mp4', fps=5)
files.download('figure47_transferring_screaming_extra3_video.mp4')


In [124]:
# %% Exercise 4
#    The target picture was initialized as random noise. But it doesn't need to be. It can be initialized to anything
#    else. Try the following target initializations: (1) the content picture; (2) the style picture; (3) a completely
#    different picture (e.g., a picture of you or a cat or the Taj Mahal).

# Quite interesting;

# Images (run cell # %% Load images first)
img_target = img_content

img_style_resized = F.interpolate(
    torch.tensor(img_style).permute(2,0,1).unsqueeze(0).float(),
    size=img_content.shape[:2],
    mode='bilinear',
    align_corners=False ).squeeze().permute(1,2,0).byte().numpy()
img_target = img_style_resized.copy()

img_target_new = imageio.v2.imread('content_image_2.jpeg')
img_content_resized = F.interpolate(
    torch.tensor(img_target_new).permute(2,0,1).unsqueeze(0).float(),
    size=img_content.shape[:2],
    mode='bilinear',
    align_corners=False ).squeeze().permute(1,2,0).byte().numpy()
img_target = img_content_resized.copy()
